In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import random

from bait.utils import common_utils, json_utils
from bait.core.bait_prompts import FILE_FORMATS, CONTEXT_SIZE, get_generate_prompt

In [ ]:
seed = GlobalCommonConfig.SEED
common_utils.set_seed(seed)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_contexts'
out_dir = f'{data_dir}/create_sft_datas'

In [ ]:
def mix_contexts(contexts_fact_dict: dict, contexts_counter_dict: dict, ext_n_fact: int, ext_n_counter: int):
    if ext_n_fact <= len(contexts_fact_dict) and ext_n_counter <= len(contexts_counter_dict):
        ext_contexts_fact = random.sample(list(contexts_fact_dict.values()), ext_n_fact)
        ext_contexts_counter = random.sample(list(contexts_counter_dict.values()), ext_n_counter)

        # 셔플 전 각각의 컨텍스트에 태그(출처)를 붙여 튜플 형태로 결합
        tagged_contexts = [(ctx, 'fact') for ctx in ext_contexts_fact] + [(ctx, 'counter') for ctx in ext_contexts_counter]

        # 태그를 붙인 상태에서 셔플
        random.shuffle(tagged_contexts)

        # 태그를 제거하고 위치 기록
        mixed_contexts = []
        fact_idxs, counter_idxs = [], []

        for i, (ctx, tag) in enumerate(tagged_contexts):
            mixed_contexts.append(ctx)
            if tag == 'fact':
                fact_idxs.append(i)
            else:
                counter_idxs.append(i)

        return mixed_contexts, fact_idxs, counter_idxs

    return None, None, None

In [ ]:
def augment_data(datas: list, zero_shot_type: str):
    datas_augmented = []

    for data in datas:
        id = data['id']
        question = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        contexts_fact = data['contexts_fact']
        contexts_counter = data['contexts_counter']

        for file_format in FILE_FORMATS:
            contexts_fact_dict = contexts_fact[file_format]
            contexts_counter_dict = contexts_counter[file_format]

            for i in range(CONTEXT_SIZE):
                ext_n_fact = i
                ext_n_counter = CONTEXT_SIZE-1-i

                mixed_contexts, fact_idxs, counter_idxs = mix_contexts(contexts_fact_dict, contexts_counter_dict, ext_n_fact, ext_n_counter)

                if mixed_contexts is not None:
                    prompt = get_generate_prompt(question, mixed_contexts)

                    sft_data = {
                        'query_id': f'{id}_{zero_shot_type}',
                        'file_format': file_format,
                        'ext_n_fact': ext_n_fact,
                        'ext_n_counter': ext_n_counter,
                        'query': question,
                        'source': {
                            'role': 'user',
                            'content': prompt
                        },
                        'target': answer_fact
                    }

                    datas_augmented.append(sft_data)

    return datas_augmented

In [ ]:
def create_sft_datas(in_file_path: str, out_prefix: str, zero_shot_type: str, train_ratio: int):
    datas = json_utils.load_json(in_file_path)
    print(f'datas size : {len(datas)}\n')

    # 컨텍스트 증강하기 전에 원본 ID를 기준으로 분할
    random.shuffle(datas)
    split_idx = int(len(datas) * train_ratio)

    datas_train = datas[:split_idx]
    datas_eval = datas[split_idx:]
    print(f'datas_train size : {len(datas_train)}')
    print(f'datas_eval size : {len(datas_eval)}\n')

    # train/eval 각각 데이터 증강
    datas_train_augmented = augment_data(datas_train, zero_shot_type)
    datas_eval_augmented = augment_data(datas_eval, zero_shot_type)
    print(f'datas_train_augmented size : {len(datas_train_augmented)}')
    print(f'datas_eval_augmented size : {len(datas_eval_augmented)}\n')

    json_utils.write_json(datas_train_augmented, f'{out_prefix}_train.json')
    json_utils.write_json(datas_eval_augmented, f'{out_prefix}_eval.json')

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B', 'Qwen2.5-3B', 'Qwen2.5-7B']
zero_shot_types = ['fact', 'counter', 'other']

for model_name in model_names:
    for zero_shot_type in zero_shot_types:
        in_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot_type}_created_contexts.json'
        out_prefix = f'{out_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot_type}_sft'

        create_sft_datas(in_file_path, out_prefix, zero_shot_type, 0.9)